<a href="https://colab.research.google.com/github/ankit-rathi/Quantvesting_v2/blob/main/myStocks_Portfolio_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
'''
!pip install pandas==2.2.2 -qq
!pip install pyxirr -qq
!pip install ta -qq
!pip install yfinance==0.2.59 -qq
'''

'\n!pip install pandas==2.2.2 -qq\n!pip install pyxirr -qq\n!pip install ta -qq\n!pip install yfinance==0.2.59 -qq\n'

In [2]:
# import libraries

import numpy as np
import pandas as pd
import datetime
import warnings
warnings.filterwarnings('ignore')
import yfinance as yfin
import ta
import matplotlib.pyplot as plt
import requests

In [3]:
# notebook start time
import time
import datetime
import pytz

start_time = time.time()

# function to print date timestamp
def print_date_time():
  tz_NY = pytz.timezone('Asia/Kolkata')
  datetime_NY = datetime.datetime.now(tz_NY)
  print("Run date time (IST):", datetime_NY.strftime("%Y-%m-%d %H:%M:%S"))

In [4]:
# function to get booked and reserved amount
def get_amt():
  # fetch the JSON data from the URL
  url = "https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioAmts.json"
  response = requests.get(url)
  pf_amts = response.json()  # parse the JSON data

  # extract values from the JSON
  py_booked_amt_dm = pf_amts["py_booked_amt_dm"]
  py_booked_amt_sv = pf_amts["py_booked_amt_sv"]
  cy_booked_amt_dm = pf_amts["cy_booked_amt_dm"]
  cy_booked_amt_sv = pf_amts["cy_booked_amt_sv"]
  reserve_amt_dm = pf_amts["reserve_amt_dm"]
  reserve_amt_sv = pf_amts["reserve_amt_sv"]

  # perform calculations
  py_booked_amt = py_booked_amt_dm + py_booked_amt_sv
  cy_booked_amt = cy_booked_amt_dm + cy_booked_amt_sv
  reserve_amt = reserve_amt_dm + reserve_amt_sv
  total_booked_amt = py_booked_amt + cy_booked_amt

  return total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt

gen_output = 0

In [5]:
# function to format the amount
def format_amt(number):
    abs_number = abs(number)

    if abs_number >= 1_00_00_000:  # Crores
        formatted_amt = f"{number / 1_00_00_000:.2f} C"
    elif abs_number >= 1_00_000:  # Lakhs
        formatted_amt = f"{number / 1_00_000:.2f} L"
    elif abs_number >= 1_000:  # Thousands
        formatted_amt = f"{number / 1_000:.2f} K"
    else:
        formatted_amt = f"{number:.2f}"

    return formatted_amt

# function to set start and end date
def get_start_end_date():
  start_date = (datetime.date.today() + datetime.timedelta(days=-365)).strftime('%Y-%m-%d')
  end_date = (datetime.date.today() + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
  return start_date, end_date

# function to fetch my portfolio csv
def get_mypfs_df():
  mypfs_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioStocks.csv')
  return mypfs_df

# function to fetch my prospects csv
def get_mypps_df():
  mypps_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myProspectsScrips.csv')
  return mypps_df

# function to fetch screener data
def get_myscreen_df():
  myscreen_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myScreenerDB.csv')
  return myscreen_df

# function to fetch momentum data
def get_myinvmt_df():
  myinvmt_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myInvestments.csv')
  return myinvmt_df

# function to get the stock ids
def get_stock_ids(df_pf):
  stock_n100 = df_pf['Symbol'].unique()

  exclude = ['CADILAHC','MMTC', 'MASFIN']

  stock_ids = df_pf[~df_pf['Symbol'].isin(exclude) ]['Symbol'].unique()

  #mypf = mypf[mypf.Forecast.notnull()]
  #stock_ids = mypf['Symbol'].unique()

  #stock_ids.sort()
  return stock_ids

# get features from screener data
def get_screener_features():
  myscreen_df = get_myscreen_df()
  cols = ['Symbol', 'EPS', 'MedPE', 'ROCE%', 'ROE%', 'CapType']
  return myscreen_df[cols]

# get relative strength
def get_relative_strength(stock_list):

    # Dictionary to store stock tickers and their corresponding percentage price change
    stock_changes = {}

    # Loop through each stock and fetch its price data
    for stock in stock_list:
        try:
            # Download the stock data for the given date range
            stock_data = yfin.Ticker(stock + '.NS').history(period='1mo', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

            # Calculate the percentage change for the stock
            if len(stock_data) > 0:
                start_price = stock_data['Close'].iloc[0]
                end_price = stock_data['Close'].iloc[-1]
                percent_change = round(((end_price - start_price) / start_price) * 100, 2)
                stock_changes[stock] = percent_change
            else:
                stock_changes[stock] = np.nan  # If no data is available, set to NaN

        except Exception as e:
            print(f"Error fetching data for {stock}: {e}")
            stock_changes[stock] = np.nan

    # Create a DataFrame with stock tickers and their percentage changes
    df = pd.DataFrame(list(stock_changes.items()), columns=['Symbol', 'Percent_Change'])

    # Drop any stocks with missing data (NaN values)
    df = df.dropna()

    # Calculate the percentile rank based on percentage change
    df['RSP'] = round(df['Percent_Change'].rank(pct=True) * 100, 2)
    cols = ['Symbol', 'RSP']
    df = df[cols]
    # Sort by percentile rank (optional)
    df = df.sort_values(by='RSP', ascending=False).reset_index(drop=True)

    return df


In [6]:
# function to get stock technicals
def stock_prec_dev(stock_symbol):
    #stock_symbol = 'ULTRACEMCO.NS'
    short_window = 20
    mid_window = 50
    long_window = 200
    moving_avg = 'SMA'

    start = datetime.datetime(*map(int, start_date.split('-')))
    end = datetime.datetime(*map(int, end_date.split('-')))

    stock_df = yfin.Ticker(stock_symbol + '.NS').history(period='max', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

    stock_df['Max'] = round(max(stock_df['Close']),2)
    stock_df = stock_df[(stock_df.index <= end_date) & (stock_df.index >= start_date)]
    stock_df['200_SMA'] = round(stock_df['Close'].rolling(window = 200, min_periods = 1).mean(),0)
    stock_df['Dev%_200'] = round((stock_df['Close'] - stock_df['200_SMA'])*100/stock_df['200_SMA'],2)
    stock_df.dropna(axis = 0, inplace = True) # remove any null rows

    stock_df['20_SMA'] = stock_df['Close'].rolling(window=20).mean()
    stock_df['50_SMA'] = stock_df['Close'].rolling(window=50).mean()
    stock_df['Symbol'] = stock_symbol

    stock_df['Close'] = round(stock_df['Close'],2)
    stock_df['Min'] = round(min(stock_df['Close']),2)
    stock_df['RSI_14'] = ta.momentum.RSIIndicator(close=stock_df['Close'], window=14).rsi()
    stock_df['RSI_14'] = round(stock_df['RSI_14'],0)
    stock_df['Prev_Close'] = stock_df['Close'].shift(1)
    stock_df.drop(['Open', 'Low', 'High', 'Volume'], axis=1, inplace=True)
    stock_df = stock_df.tail(1)

    max_SMA = max(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    min_SMA = min(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    ABS_Spread = max_SMA - min_SMA
    stock_df['Spread%'] = round((ABS_Spread / stock_df['200_SMA'].item()) * 100,2)

    return stock_df

# function to compute stock attributes
def get_common_features(stock_ids, df_mypf):

  df_prec_dev = pd.DataFrame()
  df_tmp = get_screener_features()
  df_rs = get_relative_strength(stock_ids)

  for stock_id in stock_ids:
      tmp = stock_prec_dev(stock_id)
      tmp = tmp.reset_index()
      df_prec_dev = pd.concat([df_prec_dev, tmp], ignore_index = True)
  df_prec_dev = pd.merge(df_prec_dev, df_mypf, on= 'Symbol')
  df_prec_dev = pd.merge(df_prec_dev, df_tmp, on= 'Symbol', how='left')
  df_prec_dev = pd.merge(df_prec_dev, df_rs, on= 'Symbol', how='left')
  #print(df_prec_dev.columns)
  df_prec_dev['Curr_PE'] = round(df_prec_dev['Close']/df_prec_dev['EPS'],1)
  df_prec_dev['Dev%_PE'] = round((df_prec_dev['Curr_PE'] - df_prec_dev['MedPE'])*100/df_prec_dev['MedPE'],2)
  df_prec_dev['Conviction'] = df_prec_dev['Conviction'] + '-' + df_prec_dev['CapType']
  return df_prec_dev

# function to arrange stock features
def arrange_features(df_stocks, common_cols, diff_cols):
  df_stocks_common = df_stocks[common_cols].drop_duplicates()
  df_stocks_diff = df_stocks[diff_cols]
  df_stocks_diff['Investment'] = df_stocks_diff['AvgCost'] * df_stocks_diff['Shares']
  df_stocks_diff = df_stocks_diff.groupby(['Symbol'])[['Shares', 'Investment']].aggregate(['sum']).reset_index()
  df_stocks_diff.columns = ['Symbol', 'Shares', 'Investment']
  df_stocks_diff['AvgCost'] = round(df_stocks_diff['Investment']/df_stocks_diff['Shares'],2)
  df_stocks = pd.merge(df_stocks_diff, df_stocks_common, on='Symbol')
  return df_stocks

# plot fact distribution across dimension
def plot_pie_chart(df, dimension, fact):
  # grouping the data by category and calculating the sum of fact for each type
  grouped_data = df.groupby(dimension)[fact].sum()

  # sorting the grouped data in descending order
  grouped_data = grouped_data.sort_values(ascending=False)

  # creating a pie chart
  grouped_data.plot.pie(autopct='%1.1f%%', startangle=90, figsize=(6, 6))

  # adding a title and displaying the plot
  plt.title(f'{dimension} {fact} Distribution')
  plt.ylabel('')  # To hide the y-label
  plt.show()

In [7]:
# function to get portfolio features

def get_portfolio_features(df_common_features):

  print_date_time()
  print('=================== PORTFOLIO QUALITY ====================')

  #df_common_features["Target"] = df_common_features["Target"].fillna(df_common_features["Max"])
  df_common_features['NTT'] = np.where(df_common_features['Strategy']== 'NTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['LTT'] = np.where(df_common_features['Strategy']== 'BTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['BOL'] = df_common_features['Min']

  tmp_df = df_common_features[df_common_features['Symbol'].isin(mypf_df[mypf_df['InPortfolio'] != 'NA'].Symbol.values)]
  print('qualified stocks: '+str(len(tmp_df['Symbol'].unique())))
  tmp_df1 = tmp_df[tmp_df['LatestQtr'] == 1]
  print('with latest results: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df1 = tmp_df1[tmp_df1['StarStock'] == 1]
  print('still star stocks: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df['Investment'] = tmp_df['AvgCost'] * tmp_df['Shares']
  tmp_df['Current'] = round(tmp_df['Close'] * tmp_df['Shares'],0)
  tmp_df['Previous'] = tmp_df['Prev_Close'] * tmp_df['Shares']
  tmp_df['EstimatedST'] = tmp_df['NTT'] * tmp_df['Shares']
  tmp_df['EstimatedLT'] = tmp_df['LTT'] * tmp_df['Shares']
  tmp_df['Current P/L'] = round((tmp_df['Current'] - tmp_df['Investment']),0)
  tmp_df['Today P/L%'] = round((tmp_df['Current'] - tmp_df['Previous'])*100/tmp_df['Previous'],2)
  tmp_df['Current P/L%'] = round((tmp_df['Current'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedST P/L%'] = round((tmp_df['EstimatedST'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedLT P/L%'] = round((tmp_df['EstimatedLT'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['NTT%'] = round((tmp_df['NTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['LTT%'] = round((tmp_df['LTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['Gained%'] = round((tmp_df['Close'] - tmp_df['BOL'])*100/tmp_df['BOL'],2)
  investment = round(sum(tmp_df['AvgCost']*tmp_df['Shares']),0)
  current = round(sum(tmp_df['Close']*tmp_df['Shares']),0)
  tmp_df['InitAlloc%'] = round(tmp_df['Investment']*100/investment,2)
  tmp_df['CurrAlloc%'] = round(tmp_df['Current']*100/current,2)
  tmp_df['FTT'] = tmp_df['LTT']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT'] = tmp_df['NTT']
  tmp_df['FTT%'] = tmp_df['LTT%']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT%'] = tmp_df['NTT%']
  tmp_df['FTT Amt'] = round(tmp_df['FTT%'] * tmp_df['Current']/100,0)
  tmp_df['OTT%'] = round((tmp_df['FTT'] - tmp_df['AvgCost'])*100/tmp_df['AvgCost'],2)
  tmp_df['RRR Ind'] = round(tmp_df['Current P/L']/tmp_df['FTT Amt'],2)
  tmp_df['Risk Ind'] = round(tmp_df['Current P/L%']*tmp_df['CurrAlloc%'],0)

  return tmp_df

# function to print portfolio features
def print_portfolio_stats(df_portfolio_features, myinvmt_df):
  from pyxirr import xirr

  total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt = get_amt()

  dates = myinvmt_df['Date'].values
  dates = np.append(dates, datetime.date.today().strftime('%d-%b-%y'))
  investment = myinvmt_df['Investment'].values
  dates= pd.to_datetime(dates)

  current = round(sum(df_portfolio_features['Close']*df_portfolio_features['Shares']),0) + reserve_amt
  investment_xirr = np.append(investment, current)
  cagr = round(xirr(pd.DataFrame({"dates": dates, "amounts": investment_xirr}))*100,2)

  investment = -sum(investment)
  invested = round(sum(df_portfolio_features['AvgCost']*df_portfolio_features['Shares']),0) + reserve_amt
  previous = round(sum(df_portfolio_features['Prev_Close']*df_portfolio_features['Shares']),0) + reserve_amt
  cy_invested = investment + py_booked_amt

  today_pnl_amount = current-previous
  today_pnl_percentage = round((current-previous)*100/previous,2)

  curr_pnl_amount = current-invested
  curr_pnl_percentage = round((curr_pnl_amount)*100/(cy_invested),2)

  cy_pnl_amount = cy_booked_amt + curr_pnl_amount
  cy_pnl_percentage = round((cy_pnl_amount)*100/cy_invested,2)

  overall_pnl_amount = total_booked_amt + curr_pnl_amount
  overall_pnl_percentage = round((overall_pnl_amount)*100/investment,2)

  estimate_st = round(sum(df_portfolio_features['FTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_st_pnl_amount = estimate_st-current
  est_st_pnl_percentage = round((est_st_pnl_amount)*100/current,2)

  estimate_lt = round(sum(df_portfolio_features['LTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_lt_pnl_amount = estimate_lt-current
  est_lt_pnl_percentage = round((est_lt_pnl_amount)*100/current,2)

  total_profit = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Investment']),0)
  total_loss = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Investment']),0)

  cy_booked_percentage = round((cy_booked_amt)*100/current,2)
  py_booked_percentage = round((py_booked_amt)*100/investment,2)
  total_booked_percentage = round((total_booked_amt)*100/investment,2)

  print('=================== PORTFOLIO SNAPSHOT ===================')
  print('Deployed Value: ', format_amt(investment))
  print('Current Value:  ', format_amt(current))
  print('CAGR/XIRR %:    ', '{}%'.format(cagr))
  print('Est FTT Value:  ', format_amt(estimate_st))
  print('Future Upside:  ', '{} ({}%)'.format(
      format_amt(est_st_pnl_amount),
      est_st_pnl_percentage
  ))

  print('=================== PERFORMANCE ==========================')
  print('Current PnL:             {} ({}%)'.format(
      format_amt(curr_pnl_amount), curr_pnl_percentage
  ))
  print('Total Booked PnL:        {} ({}%)'.format(
      format_amt(total_booked_amt), total_booked_percentage
  ))
  print('Booked + Current PnL:    {} ({}%)'.format(
      format_amt(overall_pnl_amount), overall_pnl_percentage
  ))
  print('CY Booked PnL:           {} ({}%)'.format(
      format_amt(cy_booked_amt), cy_booked_percentage
  ))
  print('Prev Year Booked PnL:    {} ({}%)'.format(
      format_amt(py_booked_amt), py_booked_percentage
  ))

  print('=================== TODAY / OPERATIONS ===================')
  print('Today PnL:               {} ({}%)'.format(
      format_amt(today_pnl_amount), today_pnl_percentage
  ))
  print('CY Investment:           ', format_amt(cy_invested))
  print('Reserve:                 ', format_amt(reserve_amt))

  print('=================== P&L DETAIL ============================')
  print('Total Profit:            ', format_amt(total_profit))
  print('Total Loss:              ', format_amt(total_loss))
  print('============================================================')

In [8]:
# get start and end date
start_date, end_date = get_start_end_date()

# get portfolio and prospects data
mypfs_df = get_mypfs_df()
mypps_df = get_mypps_df()
myinvmt_df = get_myinvmt_df()

# merge above datasets
mypf_df = pd.merge(mypfs_df, mypps_df, on="Symbol")

# seggregate dm and sv portfolio
dm_pf = mypf_df[mypf_df['InPortfolio'] == 'DM']
sv_pf = mypf_df[mypf_df['InPortfolio'] == 'SV']

# build portfolio stock dataframe
dm_stocks = get_stock_ids(dm_pf)
sv_stocks = get_stock_ids(sv_pf)
df_stocks = pd.concat([dm_pf,sv_pf], ignore_index = True)

# arrange common and diff stock features
common_cols = ['Symbol', 'Target', 'Criteria', 'Strategy', 'CumlRnk', 'LatestQtr', 'StarStock', 'Conviction', 'Category', 'InFolio', 'XIRR', 'MBQ']
diff_cols = ['Symbol', 'AvgCost', 'Shares']
df_stocks = arrange_features(df_stocks, common_cols, diff_cols)

# get common features
stock_ids = df_stocks['Symbol'].values
df_common_features = get_common_features(stock_ids, df_stocks)

df_common_features.reset_index(drop=True, inplace=True)
df_common_features.drop(['Date'], axis=1, inplace=True)
# get and print portfolio features
df_portfolio_features = get_portfolio_features(df_common_features)

print_portfolio_stats(df_portfolio_features, myinvmt_df)

df = df_portfolio_features
#plot_pie_chart(df, 'CapType', 'Current')
list_ox40n = list(df[df['MBQ'].str.contains('OX40N', na=False)]['Symbol'].values)

/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={
/opt

/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={
/opt

/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={
/opt

/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={
/opt

/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={
/opt

/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={
/opt

... (108 outputs left out)

/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace fo

/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace fo

/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
/opt/hostedtoolcache/Python/3.15.1/x64/lib/python3.15/site-packages/yfinance/utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace fo

Run date time (IST): 2026-11-03 22:33:00
=================== PORTFOLIO QUALITY ====================
qualified stocks: 86
with latest results: 86
still star stocks: 46
=================== PORTFOLIO SNAPSHOT ===================
Deployed Value:  1.49 C
Current Value:   1.49 C
CAGR/XIRR %:     -0.05%
Est FTT Value:   2.74 C
Future Upside:   1.25 C (84.28%)
=================== PERFORMANCE ==========================
Current PnL:             -47.71 L (-24.86%)
Total Booked PnL:        48.59 L (32.6%)
Booked + Current PnL:    87.91 K (0.59%)
CY Booked PnL:           5.71 L (3.84%)
Prev Year Booked PnL:    42.87 L (28.76%)
=================== TODAY / OPERATIONS ===================
Today PnL:               1.83 L (1.24%)
CY Investment:            1.92 C
Reserve:                  1.90 K
=================== P&L DETAIL ============================
Total Profit:             1.87 L
Total Loss:               -49.57 L


In [9]:
# top 5 near their targets
cols = ['Symbol', 'Today P/L%', 'Current P/L%', 'FTT%', 'OTT%', 'FTT Amt', 'Current P/L', 'Current', 'FTT', 'Dev%_PE', 'RSI_14','Conviction', 'Spread%',  'CumlRnk', 'RRR Ind', 'CurrAlloc%', 'Gained%', 'Criteria', 'Strategy', 'Category']
df_tmp = df_portfolio_features[~df_portfolio_features['Symbol'].isin(['ENRIN','BLUSPRING','DIGITIDE'])]
df_tmp = df_tmp.sort_values(by = 'FTT Amt', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
85,ZYDUSLIFE,0.01,30.82,3.58,35.51,9815.0,64599.0,274173.0,1285.01,29.52,69.0,H-LC,11.55,120.0,6.58,1.84,44.28,AR,ATH,PHARMA
12,BOSCH-HCIL,1.68,-24.84,33.03,-0.01,31729.0,-31743.0,96062.0,2282.00,749.92,46.0,M-SC,21.82,231.0,-1.00,0.65,63.51,OX40N,BTT,AC
76,TTKPRESTIG,-0.61,-32.19,47.61,0.09,32535.0,-32441.0,68336.0,770.00,-17.39,48.0,M-SC,5.05,250.0,-1.00,0.46,23.00,OX40N,NTT,DURABLES
40,ICICIPRULI,-0.73,-32.65,28.34,-13.56,37602.0,-64323.0,132682.0,522.35,-56.81,24.0,X-MC,8.73,78.0,-1.71,0.89,0.00,X40,ATH,INSURANCE
66,SIS,-0.95,-13.75,40.59,21.26,38871.0,-15267.0,95765.0,528.00,83.94,39.0,H-SC,8.65,168.0,-0.39,0.64,42.39,OX40N,NTT,MISC


In [10]:
# top 5 to accumulate
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 40) & (df_tmp['FTT%'] > 20)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
52,MARUTI,1.69,-13.28,44.24,25.08,115519.0,-40000.0,261118.0,17120.06,-18.78,56.0,X-LC,14.47,33.0,-0.35,1.75,6.26,XR,ATH,AUTO
38,HYUNDAI,2.27,5.00,37.08,43.94,97854.0,12568.0,263900.0,2782.73,8.72,57.0,X-LC,6.58,29.0,0.13,1.77,21.17,X40N,ATH,AUTO
51,M&M,1.37,-9.21,29.40,17.48,80365.0,-27723.0,273352.0,3762.88,-20.71,50.0,X-LC,10.16,25.0,-0.34,1.84,5.70,X40N,ATH,AUTO
35,HDFCLIFE,0.62,-8.97,54.41,40.56,154113.0,-27915.0,283244.0,811.41,-33.73,45.0,X-LC,10.27,31.0,-0.18,1.90,2.32,X40,ATH,INSURANCE
8,BAJAJHLDNG,0.01,-7.29,36.97,26.99,107253.0,-22819.0,290108.0,14191.91,-15.86,40.0,X-LC,4.15,28.0,-0.21,1.95,20.63,X40,ATH,FINANCE


In [11]:
# top 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
75,TRIDENT,7.11,-36.31,101.77,28.51,59801.0,-33494.0,58761.0,48.00,-16.02,65.0,M-SC,9.12,223.0,-0.56,0.39,16.11,XR,NTT,TEXTILES
29,GREENPANEL,6.95,-64.51,303.30,43.14,197075.0,-118101.0,64977.0,537.00,NaN,39.0,NaN,24.54,225.0,-0.60,0.44,6.95,XY24,NTT,MISC
54,NESTLEIND,4.47,5.43,23.36,30.05,70574.0,15554.0,302113.0,1540.00,-20.80,47.0,X-LC,7.55,20.0,0.22,2.03,7.51,XY25,ATH,FMCG
58,RAJOOENG,4.09,-45.87,157.25,39.24,87242.0,-47020.0,55480.0,142.72,NaN,51.0,NaN,5.01,136.0,-0.54,0.37,19.18,AR,ATH,MISC
49,KANSAINER,4.03,-37.55,81.70,13.47,137589.0,-101259.0,168408.0,340.00,-34.65,55.0,H-SC,8.66,162.0,-0.74,1.13,19.03,XY24,NTT,PAINTS


In [12]:
# bottom 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
64,SFL,-7.44,-49.91,152.80,26.63,200700.0,-130870.0,131348.0,1287.0,NaN,22.0,NaN,6.29,242.0,-0.65,0.88,9.62,XY24,NTT,MISC
24,EASEMYTRIP,-3.09,-67.11,342.95,45.70,199453.0,-118657.0,58158.0,26.4,NaN,55.0,NaN,17.81,204.0,-0.59,0.39,11.82,XY24,NTT,TRAVEL
80,VALIANTORG,-2.53,-12.73,99.55,74.15,142140.0,-20822.0,142783.0,838.0,NaN,47.0,NaN,49.53,145.0,-0.15,0.96,112.48,XR,NTT,CHEMICALS
13,BSOFT,-1.96,-41.59,154.38,48.59,126840.0,-58492.0,82161.0,819.6,NaN,58.0,NaN,11.90,172.0,-0.46,0.55,20.12,XR,ATH,IT
71,TATAELXSI,-1.77,-46.43,146.48,32.03,103443.0,-61216.0,70619.0,9161.0,-39.11,59.0,H-SC,14.32,158.0,-0.59,0.47,24.06,OX40N,NTT,IT


In [13]:
# top 5 to exit based on CumlRnk
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > -1) & (df_tmp['Current P/L%'] < 1)].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category


In [14]:
# OX40N stocks to exit
df_tmp = df_portfolio_features[(df_portfolio_features['Symbol'].isin(list_ox40n))].sort_values(by = 'Current P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,0.24,44.40,42.96,106.44,91590.0,65555.0,213199.0,1629.94,5.26,35.0,M-SC,26.22,220.0,0.72,1.43,61.54,OX40N,ATH,CABLES
66,SIS,-0.95,-13.75,40.59,21.26,38871.0,-15267.0,95765.0,528.00,83.94,39.0,H-SC,8.65,168.0,-0.39,0.64,42.39,OX40N,NTT,MISC
12,BOSCH-HCIL,1.68,-24.84,33.03,-0.01,31729.0,-31743.0,96062.0,2282.00,749.92,46.0,M-SC,21.82,231.0,-1.00,0.65,63.51,OX40N,BTT,AC
76,TTKPRESTIG,-0.61,-32.19,47.61,0.09,32535.0,-32441.0,68336.0,770.00,-17.39,48.0,M-SC,5.05,250.0,-1.00,0.46,23.00,OX40N,NTT,DURABLES
49,KANSAINER,4.03,-37.55,81.70,13.47,137589.0,-101259.0,168408.0,340.00,-34.65,55.0,H-SC,8.66,162.0,-0.74,1.13,19.03,XY24,NTT,PAINTS


In [15]:
# top 5 to book for rotation
df_tmp = df_portfolio_features[(df_portfolio_features['Current P/L%'] > 20) ].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,0.24,44.40,42.96,106.44,91590.0,65555.0,213199.0,1629.94,5.26,35.0,M-SC,26.22,220.0,0.72,1.43,61.54,OX40N,ATH,CABLES
85,ZYDUSLIFE,0.01,30.82,3.58,35.51,9815.0,64599.0,274173.0,1285.01,29.52,69.0,H-LC,11.55,120.0,6.58,1.84,44.28,AR,ATH,PHARMA


In [16]:
# top 5 to monitor for exit
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > 0) ].sort_values(by = 'Dev%_PE', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,0.24,44.40,42.96,106.44,91590.0,65555.0,213199.0,1629.94,5.26,35.0,M-SC,26.22,220.0,0.72,1.43,61.54,OX40N,ATH,CABLES
67,STARHEALTH,2.06,1.12,45.63,47.26,129711.0,3144.0,284267.0,761.00,-10.89,42.0,H-SC,5.37,173.0,0.02,1.91,22.91,XY24,NTT,INSURANCE


In [17]:
# top 5 to book for rotation from weak conviction
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[~df_tmp['Criteria'].isin(['XY25','XY24','X40', 'X40N'])].sort_values(by = 'RRR Ind', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,0.24,44.40,42.96,106.44,91590.0,65555.0,213199.0,1629.94,5.26,35.0,M-SC,26.22,220.0,0.72,1.43,61.54,OX40N,ATH,CABLES
84,WIPRO,0.81,-9.01,112.12,93.01,255297.0,-22540.0,227700.0,420.00,-23.12,72.0,M-LC,8.83,98.0,-0.09,1.53,26.28,XR,NTT,IT
80,VALIANTORG,-2.53,-12.73,99.55,74.15,142140.0,-20822.0,142783.0,838.00,NaN,47.0,NaN,49.53,145.0,-0.15,0.96,112.48,XR,NTT,CHEMICALS
70,TANLA,2.57,-25.18,187.58,115.17,380013.0,-68174.0,202587.0,1903.93,NaN,79.0,NaN,9.71,147.0,-0.18,1.36,79.09,AR,ATH,IT
3,ALKYLAMINE,0.48,-29.35,172.54,92.54,123069.0,-29635.0,71328.0,4520.85,NaN,40.0,NaN,9.86,142.0,-0.24,0.48,35.45,SR,ATH,CHEMICALS


In [18]:
# top 5 to accumulate based on lowest RSI_14
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'RSI_14', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
37,HONAUT,-0.48,-24.63,100.18,50.88,145612.0,-47488.0,145350.0,58192.79,-41.15,21.0,X-SC,5.93,90.0,-0.33,0.98,10.74,X40N,ATH,ELECTRICAL
40,ICICIPRULI,-0.73,-32.65,28.34,-13.56,37602.0,-64323.0,132682.0,522.35,-56.81,24.0,X-MC,8.73,78.0,-1.71,0.89,0.00,X40,ATH,INSURANCE
6,AWL,-0.89,-43.42,195.21,67.02,393512.0,-154725.0,201584.0,485.00,-58.24,32.0,X-SC,6.39,81.0,-0.39,1.35,0.21,XY24,NTT,FMCG
7,BAJAJHFL,0.01,-30.16,132.13,62.13,251979.0,-82341.0,190705.0,181.50,-41.76,39.0,X-MC,5.09,60.0,-0.33,1.28,6.98,X40N,ATH,FINANCE
8,BAJAJHLDNG,0.01,-7.29,36.97,26.99,107253.0,-22819.0,290108.0,14191.91,-15.86,40.0,X-LC,4.15,28.0,-0.21,1.95,20.63,X40,ATH,FINANCE


In [19]:
# top 5 to accumulate based on Spread and CumlRnk
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 100)].sort_values(by = 'Spread%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
55,PAGEIND,-0.62,-7.98,44.38,32.85,109601.0,-21419.0,246960.0,50936.07,-31.61,48.0,X-MC,1.33,55.0,-0.20,1.66,17.93,X40,ATH,APPARELS
1,ABBOTINDIA,-0.08,-13.45,33.30,15.37,51548.0,-24055.0,154800.0,34390.26,-29.45,48.0,X-MC,1.64,58.0,-0.47,1.04,4.86,X40,ATH,PHARMA
17,CIPLA,0.71,5.00,17.99,23.89,56794.0,15044.0,315699.0,1648.14,19.26,64.0,X-LC,2.20,18.0,0.26,2.12,18.25,X40N,ATH,PHARMA
11,BERGEPAINT,2.37,-11.60,41.76,25.32,95154.0,-29891.0,227858.0,680.00,-23.12,58.0,X-MC,2.34,68.0,-0.31,1.53,19.49,XY24,NTT,PAINTS
33,HCLTECH,0.36,-1.53,27.65,25.70,98284.0,-5515.0,355458.0,1837.07,-6.32,72.0,X-LC,3.97,8.0,-0.06,2.39,41.82,X40,ATH,IT


In [20]:
# top 5 to accumulate based on lowest Gained%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'Gained%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
40,ICICIPRULI,-0.73,-32.65,28.34,-13.56,37602.0,-64323.0,132682.0,522.35,-56.81,24.0,X-MC,8.73,78.0,-1.71,0.89,0.00,X40,ATH,INSURANCE
6,AWL,-0.89,-43.42,195.21,67.02,393512.0,-154725.0,201584.0,485.00,-58.24,32.0,X-SC,6.39,81.0,-0.39,1.35,0.21,XY24,NTT,FMCG
28,GODREJCP,1.03,-24.50,76.02,32.90,143229.0,-61127.0,188410.0,1460.97,-25.90,41.0,X-MC,16.86,32.0,-0.43,1.27,1.47,X40N,ATH,FMCG
35,HDFCLIFE,0.62,-8.97,54.41,40.56,154113.0,-27915.0,283244.0,811.41,-33.73,45.0,X-LC,10.27,31.0,-0.18,1.90,2.32,X40,ATH,INSURANCE
19,COLPAL,3.60,-38.04,122.35,37.76,199638.0,-100195.0,163170.0,3628.04,-25.02,42.0,X-MC,14.09,62.0,-0.50,1.10,3.76,XY25,ATH,FMCG


In [21]:
# top 5 to accumulate based on lowest CurrAlloc%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
56,PGHH,2.49,-47.12,147.78,31.02,166850.0,-100616.0,112904.0,17484.40,-61.09,51.0,X-SC,24.92,57.0,-0.60,0.76,6.29,X40,ATH,FMCG
40,ICICIPRULI,-0.73,-32.65,28.34,-13.56,37602.0,-64323.0,132682.0,522.35,-56.81,24.0,X-MC,8.73,78.0,-1.71,0.89,0.00,X40,ATH,INSURANCE
2,ACC,1.61,-47.63,233.82,74.81,328311.0,-127716.0,140412.0,3906.00,-48.10,49.0,X-SC,16.49,85.0,-0.39,0.94,4.02,XY24,BTT,CEMENT
37,HONAUT,-0.48,-24.63,100.18,50.88,145612.0,-47488.0,145350.0,58192.79,-41.15,21.0,X-SC,5.93,90.0,-0.33,0.98,10.74,X40N,ATH,ELECTRICAL
1,ABBOTINDIA,-0.08,-13.45,33.30,15.37,51548.0,-24055.0,154800.0,34390.26,-29.45,48.0,X-MC,1.64,58.0,-0.47,1.04,4.86,X40,ATH,PHARMA


In [22]:
# top 5 to accumulate based on CumlRnk
df_tmp = df_portfolio_features.sort_values(by = 'CumlRnk', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
47,ITC,2.82,-22.01,65.02,28.70,221900.0,-96316.0,341279.0,452.00,-29.53,62.0,X-LC,8.51,1.0,-0.43,2.29,7.41,X40,NTT,FMCG
72,TCS,-0.10,-24.73,68.39,26.75,251243.0,-120707.0,367368.0,4208.34,-38.95,67.0,X-LC,5.19,2.0,-0.48,2.47,27.42,X40,ATH,IT
45,INFY,1.15,-16.94,71.76,42.66,238924.0,-67906.0,332949.0,1972.00,-39.69,61.0,X-LC,9.18,3.0,-0.28,2.24,19.19,X40,NTT,IT
74,TMPV,1.58,-32.65,116.84,46.05,352069.0,-146046.0,301326.0,600.00,-87.07,47.0,X-LC,17.81,4.0,-0.41,2.02,5.93,XY24,NTT,AUTO
33,HCLTECH,0.36,-1.53,27.65,25.70,98284.0,-5515.0,355458.0,1837.07,-6.32,72.0,X-LC,3.97,8.0,-0.06,2.39,41.82,X40,ATH,IT


In [23]:
# top 5 for average up
df_tmp = df_portfolio_features[(df_portfolio_features['Dev%_200'] > 5)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
12,BOSCH-HCIL,1.68,-24.84,33.03,-0.01,31729.0,-31743.0,96062.0,2282.00,749.92,46.0,M-SC,21.82,231.0,-1.00,0.65,63.51,OX40N,BTT,AC
80,VALIANTORG,-2.53,-12.73,99.55,74.15,142140.0,-20822.0,142783.0,838.00,NaN,47.0,NaN,49.53,145.0,-0.15,0.96,112.48,XR,NTT,CHEMICALS
70,TANLA,2.57,-25.18,187.58,115.17,380013.0,-68174.0,202587.0,1903.93,NaN,79.0,NaN,9.71,147.0,-0.18,1.36,79.09,AR,ATH,IT
25,FINCABLES,0.24,44.40,42.96,106.44,91590.0,65555.0,213199.0,1629.94,5.26,35.0,M-SC,26.22,220.0,0.72,1.43,61.54,OX40N,ATH,CABLES
85,ZYDUSLIFE,0.01,30.82,3.58,35.51,9815.0,64599.0,274173.0,1285.01,29.52,69.0,H-LC,11.55,120.0,6.58,1.84,44.28,AR,ATH,PHARMA


In [24]:
# top 5 RSP
df_tmp = df_portfolio_features.sort_values(by = 'RSP', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
70,TANLA,2.57,-25.18,187.58,115.17,380013.0,-68174.0,202587.0,1903.93,NaN,79.0,NaN,9.71,147.0,-0.18,1.36,79.09,AR,ATH,IT
84,WIPRO,0.81,-9.01,112.12,93.01,255297.0,-22540.0,227700.0,420.00,-23.12,72.0,M-LC,8.83,98.0,-0.09,1.53,26.28,XR,NTT,IT
33,HCLTECH,0.36,-1.53,27.65,25.70,98284.0,-5515.0,355458.0,1837.07,-6.32,72.0,X-LC,3.97,8.0,-0.06,2.39,41.82,X40,ATH,IT
71,TATAELXSI,-1.77,-46.43,146.48,32.03,103443.0,-61216.0,70619.0,9161.00,-39.11,59.0,H-SC,14.32,158.0,-0.59,0.47,24.06,OX40N,NTT,IT
72,TCS,-0.10,-24.73,68.39,26.75,251243.0,-120707.0,367368.0,4208.34,-38.95,67.0,X-LC,5.19,2.0,-0.48,2.47,27.42,X40,ATH,IT


In [25]:
# Top N allocation
df_tmp = df_portfolio_features.sort_values(by = 'CurrAlloc%', ascending=False)
top_n_values = [10, 25, 50]

sum_df = pd.DataFrame({
    'Top_N': top_n_values,
    'Sum_Alloc%': [df_tmp['CurrAlloc%'].head(n).sum() for n in top_n_values]
})

sum_df

,Top_N,Sum_Alloc%
0,10,22.28
1,25,49.58
2,50,80.41


In [26]:
# market-cap-wise allocation
df_tmp = df_portfolio_features[cols]
df_tmp.groupby(df_tmp['Conviction'].str[-2:])['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
LC    42.84
MC    20.59
SC    19.90
Name: CurrAlloc%, dtype: float64

In [27]:
# criteria-wise allocation
df_portfolio_features.groupby('Criteria')['CurrAlloc%'].sum().sort_values(ascending=False)

Criteria
X40      27.59
XY24     22.14
X40N     18.49
XR        9.80
XY25      9.54
OX40N     6.53
AR        5.11
SR        0.79
Name: CurrAlloc%, dtype: float64

In [28]:
# conviction-wise allocation
df_portfolio_features.groupby('Conviction')['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
X-LC    39.47
X-MC    18.23
H-SC     7.92
X-SC     7.01
M-SC     4.97
H-LC     1.84
M-LC     1.53
M-MC     1.05
H-MC     0.98
L-MC     0.33
Name: CurrAlloc%, dtype: float64

In [29]:
# sector-wise stats
df_tmp = df_portfolio_features.groupby('Category')[['CurrAlloc%', 'Current', 'Current P/L', 'FTT Amt']].sum().sort_values(by=['Current', 'Current P/L'], ascending=False)
df_tmp['Current P/L%'] = round(df_tmp['Current P/L'] * 100 / df_tmp['Current'], 2)
df_tmp['FTT%'] = round(df_tmp['FTT Amt'] * 100 / df_tmp['Current'], 2)
cols = ['CurrAlloc%', 'Current P/L%', 'FTT%']
df_tmp[cols].sort_values(by=['CurrAlloc%'], ascending=False)

,CurrAlloc%,Current P/L%,FTT%
Category,,,
FMCG,17.79,-29.60,76.22
IT,13.22,-30.37,98.51
FINANCE,10.37,-30.50,86.25
AUTO,8.26,-21.68,65.09
PHARMA,6.38,6.55,19.20
INSURANCE,5.84,-13.81,44.77
MISC,5.39,-53.50,115.20
PAINTS,5.35,-29.75,52.65
BANKS,4.90,-22.92,84.57


In [30]:
# money to be made criteria-wise
df_portfolio_features.groupby('Criteria')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Criteria,,
XY24,4134892.0,19
X40,2286331.0,17
XR,1595382.0,13
X40N,1589800.0,12
AR,1033321.0,7
OX40N,817091.0,10
XY25,795056.0,6
SR,296292.0,2


In [31]:
# money to be made conviction-wise
df_portfolio_features.groupby('Conviction')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Conviction,,
X-LC,3068105.0,19
X-MC,1695594.0,14
X-SC,1230717.0,6
H-SC,970002.0,8
M-SC,422602.0,6
H-MC,267816.0,1
M-LC,255297.0,1
M-MC,88831.0,1
L-MC,53767.0,1


In [32]:
# money to be made criteria and conviction-wise
df_portfolio_features.groupby(['Conviction', 'Criteria'])['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

sum  count
Conviction Criteria                  
X-LC       X40       1348610.0      8
X-SC       XY24       721823.0      2
X-LC       X40N       622938.0      5
           XY24       595954.0      2
H-SC       XY24       593184.0      3
X-MC       X40N       566742.0      4
           X40        541191.0      6
X-LC       XY25       385084.0      3
X-MC       XY25       321141.0      2
X-SC       X40        276497.0      2
H-MC       XY24       267816.0      1
X-MC       XY24       266520.0      2
M-LC       XR         255297.0      1
H-SC       OX40N      253743.0      3
X-SC       X40N       232397.0      2
M-SC       OX40N      223116.0      4
           XY24       139685.0      1
H-SC       XR         123075.0      2
X-LC       XR         115519.0      1
M-MC       XY25        88831.0      1
M-SC       XR          59801.0      1
L-MC       XR          53767.0      1
H-LC       AR           9815.0      1

In [33]:
# notebook execution time

end_time = time.time()
execution_time = round(end_time - start_time, 0)
print(f"Notebook execution time: {execution_time} seconds")

Notebook execution time: 37.0 seconds
